# PoseFly Rolling Shutter Test

1- Imports

In [78]:
import time
import cv2
import numpy as np
import math

2- Global ISO / Hz + Rolling Shutter Function

In [79]:
ISO = 800
SHUTTER_HZ = 6000
BASE_BRIGHTNESS = 10

def apply_rolling_shutter(frame, iso=ISO, shutter_hz=SHUTTER_HZ):
    h, w, _ = frame.shape
    row_time = 1.0 / max(1, int(shutter_hz))
    out = frame.copy()

    # --- Camera-like mapping (matches camera.py) ---

    # ISO -> gain (log mapping)
    iso_f = float(max(1, iso))
    t_iso = (math.log(iso_f) - math.log(50.0)) / (math.log(6400.0) - math.log(50.0))
    t_iso = max(0.0, min(1.0, t_iso))
    gain = 2.0 + t_iso * 18.0  # ~2..20

    # shutter_hz -> exposure proxy (higher Hz => darker)
    sh_f = float(max(1, shutter_hz))
    t_sh = (math.log(sh_f) - math.log(5.0)) / (math.log(6000.0) - math.log(5.0))
    t_sh = max(0.0, min(1.0, t_sh))

    exposure = -10.0 + (1.0 - t_sh) * 6.0
    exposure_scale = 2.0 ** (exposure / 2.0)

    brightness = 95.0 + (1.0 - t_sh) * 15.0
    brightness_scale = brightness / 100.0

    # --- OOK stripe model (row-time based) ---
    led_freq_hz = 2000
    duty = 0.5
    contrast = 0.90

    y = np.arange(h, dtype=np.float32)
    t = y * row_time
    phase = (t * led_freq_hz) % 1.0
    on = (phase < duty).astype(np.float32)

    row_gain = (1.0 - contrast) + contrast * on
    row_gain = row_gain[:, None]

    # Apply everything in HSV value channel
    hsv = cv2.cvtColor(out, cv2.COLOR_BGR2HSV).astype(np.float32)
    v = hsv[:, :, 2]

    v *= gain
    v *= exposure_scale
    v *= brightness_scale
    v *= row_gain

    hsv[:, :, 2] = np.clip(v, 0, 255)
    out = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    return out

3- Shutter apply function (Camera Version)

In [80]:
import cv2
import math

import cv2
import math

def apply_rolling_shutter_hardware(cap, iso: int, shutter_hz: float):
    global GLOBAL_BRIGHTNESS

    if cap is None:
        return

    iso = int(max(50, min(6400, iso)))
    shutter_hz = float(max(5.0, min(6000.0, shutter_hz)))

    # Prefer manual exposure (DirectShow convention)
    cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.75)

    # ISO -> gain (log mapping)
    t_iso = (math.log(iso) - math.log(50.0)) / (math.log(6400.0) - math.log(50.0))
    t_iso = max(0.0, min(1.0, t_iso))
    gain = 2.0 + t_iso * 18.0  # ~2..20
    cap.set(cv2.CAP_PROP_GAIN, float(gain))

    # shutter_hz -> exposure (higher Hz => shorter exposure => darker)
    t_sh = (math.log(shutter_hz) - math.log(5.0)) / (math.log(6000.0) - math.log(5.0))
    t_sh = max(0.0, min(1.0, t_sh))
    exposure = -10.0 + (1.0 - t_sh) * 6.0  # ~[-10..-4]
    cap.set(cv2.CAP_PROP_EXPOSURE, float(exposure))

    # brightness compensation (GLOBAL)
    GLOBAL_BRIGHTNESS = BASE_BRIGHTNESS + (1.0 - t_sh)
    cap.set(cv2.CAP_PROP_BRIGHTNESS, float(GLOBAL_BRIGHTNESS))


4- Capture One Photo (Original)

In [81]:
CAM_INDEX = 0

Previous_ISO = 250
Previous_SHUTTER_HZ = 1000

cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera index {CAM_INDEX}")

# Apply rolling-shutter camera settings
apply_rolling_shutter_hardware(cap, Previous_ISO, Previous_SHUTTER_HZ)

# Let the camera settle
for _ in range(8):
    cap.read()

print("Press SPACE to capture, ESC to cancel.")

captured = None
while True:
    ok, frame = cap.read()
    if not ok or frame is None:
        continue

    cv2.imshow("Live Preview (SPACE=capture)", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == 27:  # ESC
        break
    if key == 32:  # SPACE
        captured = frame.copy()
        break

cap.release()
cv2.destroyAllWindows()

if captured is None:
    raise RuntimeError("No photo captured.")
else:
    print("Captured one frame:", captured.shape)

Press SPACE to capture, ESC to cancel.
Captured one frame: (480, 640, 3)


5- Apply Rolling Shutter + Save Both

In [82]:
import cv2
import numpy as np

def add_label_below(img, label, font_scale=0.8, thickness=2, pad=12):
    h, w = img.shape[:2]
    label_h = 40

    canvas = np.zeros((h + label_h, w, 3), dtype=np.uint8)
    canvas[:h] = img

    text_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    text_x = (w - text_size[0]) // 2
    text_y = h + label_h - pad

    cv2.putText(
        canvas,
        label,
        (text_x, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )
    return canvas


# Original frame
before = captured

# Rolling shutter output
after = apply_rolling_shutter(
    before,
    iso=ISO,
    shutter_hz=SHUTTER_HZ
)

before_labeled = add_label_below(
    before,
    f"Regular Camera | ISO={Previous_ISO}, Shutter={Previous_SHUTTER_HZ} Hz"
)

after_labeled = add_label_below(
    after,
    f"Rolling Shutter | ISO={ISO}, Shutter={SHUTTER_HZ} Hz"
)

6 — Show Before vs After (Side-by-Side)

In [83]:
combo = np.hstack([before_labeled, after_labeled])

cv2.imshow("Rolling Shutter Comparison (s = save, q = quit)", combo)

key = cv2.waitKey(0) & 0xFF

if key == ord('s'):
    from pathlib import Path
    out_dir = Path("rolling_shutter_outputs")
    out_dir.mkdir(parents=True, exist_ok=True)

    save_path = out_dir / f"comparison_iso{ISO}_hz{SHUTTER_HZ}.jpg"
    cv2.imwrite(str(save_path), combo)
    print("Saved:", save_path)

cv2.destroyAllWindows()
